In [1]:
import os
import pandas as pd
import xmltodict
from pathlib import Path

In [2]:
# os.chdir("/Users/wes/Desktop/MSc-Data-Science/MSc-Data-Science/MAST7865 - Data Science Project/plan/data")
os.getcwd()

'e:\\Users\\wesle\\biomedical_kg_thesis\\exploration'

In [3]:
train_xml_path = "../data/raw/BioRED/Train.BioC.XML"
test_xml_path = "../data/raw/BioRED/Test.BioC.XML"
dev_xml_path = "../data/raw/BioRED/Dev.BioC.XML"

xml_paths = (train_xml_path, test_xml_path, dev_xml_path)

In [4]:
def load_xml_dict(filepath: str | Path) -> dict:
    """Load an XML file and convert it to a dictionary."""

    filepath = Path(filepath)

    with filepath.open("r", encoding="utf-8") as file:
        return xmltodict.parse(file.read())

In [5]:
train_xml_dict = load_xml_dict(train_xml_path)
test_xml_dict = load_xml_dict(test_xml_path)
dev_xml_dict = load_xml_dict(dev_xml_path)

In [6]:
def _ensure_list(value):
    """xmltodict returns a dict instead of a list when an element occurs
    only once. Normalise to a list so we can always iterate safely."""
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def _infons_to_dict(infon) -> dict:
    """Convert BioC <infon key="..">text</infon> elements into a plain dict."""
    return {item["@key"]: item.get("#text") for item in _ensure_list(infon)}


def construct_corpus(xml_dict: dict, split: str) -> pd.DataFrame:
    """
    Convert a parsed BioRED BioC XML collection into a document dataframe.
    """
    pmids, titles, abstracts = [], [], []

    for paper in _ensure_list(xml_dict["collection"]["document"]):
        title = None
        abstract = None

        for passage in _ensure_list(paper["passage"]):
            passage_type = passage["infon"]["#text"]

            if passage_type == "title":
                title = passage.get("text", "")
            elif passage_type == "abstract":
                abstract = passage.get("text", "")

        pmids.append(paper["id"])
        titles.append(title)
        abstracts.append(abstract)

    return pd.DataFrame({
        "pmid": pmids,
        "title": titles,
        "abstract": abstracts,
        "split": split,
    })


def extract_entities(xml_dict: dict, split: str) -> pd.DataFrame:
    """
    Convert a parsed BioRED BioC XML collection into an entity-mention
    dataframe: one row per annotation, across all passages of each document.
    """
    rows = []

    for paper in _ensure_list(xml_dict["collection"]["document"]):
        pmid = paper["id"]

        for passage in _ensure_list(paper["passage"]):
            for annotation in _ensure_list(passage.get("annotation")):
                infons = _infons_to_dict(annotation["infon"])
                location = annotation["location"]
                start = int(location["@offset"])
                length = int(location["@length"])

                rows.append({
                    "pmid": pmid,
                    "entity_id": annotation["@id"],
                    "text": annotation.get("text", ""),
                    "entity_type": infons.get("type"),
                    "identifier": infons.get("identifier"),
                    "start": start,
                    "end": start + length,
                })

    df = pd.DataFrame(rows, columns=["pmid", "entity_id", "text", "entity_type", "identifier", "start", "end"])
    df["split"] = split
    return df


def extract_relations(xml_dict: dict, split: str) -> pd.DataFrame:
    """
    Convert a parsed BioRED BioC XML collection into a relation dataframe:
    one row per relation, at document level. entity1/entity2 identifiers
    reference normalised entity identifiers, not per-mention entity_id.
    """
    rows = []

    for paper in _ensure_list(xml_dict["collection"]["document"]):
        pmid = paper["id"]

        for relation in _ensure_list(paper.get("relation")):
            infons = _infons_to_dict(relation["infon"])

            rows.append({
                "pmid": pmid,
                "relation_id": relation["@id"],
                "entity1_identifier": infons.get("entity1"),
                "entity2_identifier": infons.get("entity2"),
                "relation_type": infons.get("type"),
            })

    df = pd.DataFrame(rows, columns=["pmid", "relation_id", "entity1_identifier", "entity2_identifier", "relation_type"])
    df["split"] = split
    return df


def _representative_entities(entities_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a (pmid, identifier) -> (text, entity_type) lookup for resolving
    relations back to entity text/type.

    Some annotations carry a *composite* identifier -- several IDs joined
    by commas, e.g. "836,840" for a mention like "caspase-3/7" that refers
    to two genes at once. Relations reference the individual component IDs,
    so identifiers are split and exploded before the lookup is built,
    otherwise a relation pointing at just "840" would fail to match the
    combined "836,840" annotation. Where several mentions share the same
    identifier, the first one encountered is used as the representative.
    """
    exploded = entities_df.copy()
    exploded["identifier"] = exploded["identifier"].str.split(",")
    exploded = exploded.explode("identifier")
    return (
        exploded
        .drop_duplicates(subset=["pmid", "identifier"], keep="first")
        [["pmid", "identifier", "text", "entity_type"]]
    )


def build_entity_relations(entities_df: pd.DataFrame, relations_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge entity mentions and relations into a single entity-relationship
    dataframe: one row per relation, with entity text/type resolved for
    both sides via their normalised identifier.
    """
    lookup = _representative_entities(entities_df)

    merged = relations_df.merge(
        lookup, left_on=["pmid", "entity1_identifier"], right_on=["pmid", "identifier"], how="left"
    ).rename(columns={"text": "entity_1", "entity_type": "entity_1_type"}).drop(columns="identifier")

    merged = merged.merge(
        lookup, left_on=["pmid", "entity2_identifier"], right_on=["pmid", "identifier"], how="left"
    ).rename(columns={"text": "entity_2", "entity_type": "entity_2_type"}).drop(columns="identifier")

    merged = merged.rename(columns={
        "entity1_identifier": "entity_1_identifier",
        "entity2_identifier": "entity_2_identifier",
        "relation_type": "relation",
    })

    return merged[[
        "pmid", "entity_1", "entity_1_type", "entity_1_identifier",
        "relation", "entity_2", "entity_2_type", "entity_2_identifier", "split"
    ]]

In [7]:
for split, xml_dict in [("train", train_xml_dict), ("test", test_xml_dict), ("dev", dev_xml_dict)]:
    documents_df = construct_corpus(xml_dict, split)
    entities_df = extract_entities(xml_dict, split)
    relations_df = extract_relations(xml_dict, split)
    entity_relations_df = build_entity_relations(entities_df, relations_df)

    documents_df.set_index("pmid", inplace=True)
    documents_df.to_csv(f"../data/processed/biored/br_{split}.csv")
    entities_df.to_csv(f"../data/processed/biored/br_{split}_entities.csv", index=False)
    # relations_df.to_csv(f"../data/processed/biored/br_{split}_relations.csv", index=False)
    entity_relations_df.to_csv(f"../data/processed/biored/br_{split}_entity_relations.csv", index=False)